In [2]:
import pandas as pd
import numpy as np

# fetching the data directly from UCI
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/abalone/abalone.data"
column_names = [
    "Sex", "Length", "Diameter", "Height", "Whole_weight",
    "Shucked_weight", "Viscera_weight", "Shell_weight", "Rings"
]
df = pd.read_csv(url, names=column_names)

# creating the Age variable (defined from UCI)
df['Age'] = df['Rings'] + 1.5

# Market Value Proxy
diam_75 = df['Diameter'].quantile(0.75)
weight_75 = df['Whole_weight'].quantile(0.75)
diam_25 = df['Diameter'].quantile(0.25)
weight_25 = df['Whole_weight'].quantile(0.25)

# logical conditions for each market tier
conditions = [
    (df['Diameter'] >= diam_75) & (df['Whole_weight'] >= weight_75), # premium
    (df['Diameter'] >= diam_25) & (df['Whole_weight'] >= weight_25)  # standard
]

# assigning base prices for premium, standard, and sub-standard
base_prices = [50.0, 30.0, 15.0]

# applying the conditions to create a base price column
df['Base_Price'] = np.select(conditions, base_prices[:-1], default=base_prices[-1])

# noise: epsilon ~ N(0, sigma^2)
# using a standard deviation of 3 to represent realistic price fluctuation
np.random.seed(42)
noise = np.random.normal(loc=0.0, scale=3.0, size=len(df))

# calculating final market value, rounding it, and clipping at 0 to prevent negative prices
df['Market_Value'] = np.round(df['Base_Price'] + noise, 2)
df['Market_Value'] = df['Market_Value'].clip(lower=0)

# dropping the intermediate base price column to keep the dataframe clean
df.drop(columns=['Base_Price'], inplace=True)

# organizing and separating the data for specific modeling tasks
targets = df[['Age', 'Market_Value']]
pca_features = df[['Whole_weight', 'Shucked_weight', 'Viscera_weight', 'Shell_weight']]
remaining_predictors = df[['Sex', 'Length', 'Diameter', 'Height']]

# Verify the output
print("Data preview:")
print(df[['Diameter', 'Whole_weight', 'Age', 'Market_Value']].head())

URLError: <urlopen error [SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1010)>